# API RAG QA Pipeline

This notebook builds a RAG pipeline for the PDF-derived Markdown corpus and answers the questions in `qa_set.csv`.

It is designed for a non-local/cloud notebook environment:

- Put the converted Markdown files and `qa_set.csv` in the same folder as this notebook.
- Set `CSCS_API_KEY` in the environment, or create a file named `environment` containing `CSCS_API_KEY=...`.
- The LLM answer generation uses the Swiss AI OpenAI-compatible API.

Retrieval settings used below:

- Chunk size: `250` words
- Chunk overlap: `50` words
- Embedding model: `Snowflake/snowflake-arctic-embed-l-v2.0`
- Cosine similarity threshold: `0.30`
- Maximum context chunks sent to the model: `6`
- Minimum fallback chunks: `3`


## 1. Install and Import Dependencies

In [ ]:
# Run this cell once in a fresh notebook environment.
%pip install -q openai python-dotenv pandas numpy tqdm

In [35]:
import csv
import json
import os
import re
import sqlite3
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv(dotenv_path="environment")

WORK_DIR = Path.cwd()
QA_CSV = WORK_DIR / "qa_set.csv"
OUTPUT_CSV = WORK_DIR / "qa_set_api_rag_answers.csv"
DB_PATH = WORK_DIR / "rag_api_embedding_database.sqlite"

EMBEDDED_QA_CSV = """question;answer
What year where there most casualties from man-made disasters in the recorded data?;2002. In that year more than 10,000 casualties are ascribed to man-made catastrophes.
In what year did the quantity of man-made disasters peak in the recorded data between 1970 and 2023?;Man-made disasters peaked in 2005
Between 1970 and 2023 what year in the data shows the largest number of natural catastrophes?;2023.  There were 218 instances of natural catastrophes.
What year between 1994 and 2023 had the most high severity ($5 billion in damages or more) natural catastrophes?;2011 with 6.
How much higher are the 2023 insured losses than the previous 10 year average?;They are higher by 21%.
Which figure shows the trend in insured losses over data from 1994 to 2023? In this figure, what is the highest insured loss year on record?;
Before 2023, what was the highest year on record for European Severe Convective Storm losses?;
What regions does the Swiss Re report on natural catastrophes split the US into for severe convective storm risk?;
What is the highest Benefit to Cost ratio building code element described in the Swiss Re report on natural catastrophes?;
According to the Swiss Re Institute report on natural catastrophes, 2017 was a standout year in terms of insured loss damages. What were the names of the weather events which contributed most to this figure?;
Given how the Swiss Re Institute classifies primary and secondary perils, to which category can we attribute more losses in 2023? How is insurance claim tracking characterized in this category?;
In historical data from the Swiss Re Institute, which geographical grouping of countries has the smallest proportion of insured losses to uninsured losses between 2014 and 2023?;
What is the lower bound of dead or missing which have to be reported in connection to a natural catastrophe in order for the Swiss Re data to report it in their statistics?;
Excluding the overall UN average, which group of nations have more than 100 mobile broadband subscriptions per 100 residents in 2024?;
According to survey data, which feature of government web-portals experienced the largest between 2022 and 2024?;
When were the simplified four stages of E-government adoption published? Which revision of the EGDI is this related to?;
Between what years was the EGDI revision 3.0 acive?;
Given when academic articles using the term started being published, when were the terms E-government development index and Online services index introduced?;
Of the questions highlighted from the Member States Questionnaire in a chart, is there a discontinuity in the numbering of the questions featured? If so, which numbers are missing?;
What was the percentage increase over all the 193 UN member states in EGDI scores between 2022 and 2024?;
Of the countries with very high OSI levels and high EGDI divergence what is the one with the lowest Telecommunications Infrasructure Index?;
What grouping of nations has the closest OSI subindex average to the overall UN 193 average?;
Why did the average number of provided online services increase while the percentage stayed the same between 2022 and 2024? How many services were assessed in each of the 2 years?;
What percentage of european countries support filing income tax online?;
In what percentage of countries in Oceania is one able to apply for a death certificate through a fully digitized process?;
What region of the world contains the countries that enable fully digitized vehicle regitration? What is the percentage of countries in this region that support this?;
In what percentage of countries in the Americas is one able to apply for disability compensation in at least a partially online manner?;
What number of countries in Africa support digital invoicing? What is the percentage increase in that figure since 2022?;
What number of countries in Europe offer an E-procurement platform? Not percentage, number of countries.;
What fully digital service for individuals in vulnerable situations experienced the largest percentge decline between 2022 and 2024?;
What service for individuals in vulnerable situations has no fully digitized component in Oceania?;
What proportion of countries offer judiciary services in a way that is accessible on mobile or through an app?;
What landlocked countries have moved from high to very high E-government development index in the survey period?;
What small island nation moved from the middle to the high EGDI group in the last survey period?;
How has Japan named their initiative for removing bureaucratic inefficiencies and improving their digital government tools? What is the project's initial budget?;
What two departments of the UK govenrment have been merged in the effort to enhance their digital transformation?;
What country grouping appears to have the most drastic difference between EGDI levels of it's constituents?;
What is the change in percentage of cities providing information on procurement between 2022 and 2024?;
In the year before the two investment segments equalized, what was the amount invested in energy transition power generation vs fossil fuel power generation?;
How many of the top 10 costliest environmental disasters of 2023 occurred in Latin America? Which country faced the msot expensive one?;
"""

CHUNK_WORDS = 250
OVERLAP_WORDS = 50
SIMILARITY_THRESHOLD = 0.30
MAX_CONTEXT_CHUNKS = 6
MIN_CONTEXT_CHUNKS = 3

API_BASE_URL = "https://api.swissai.svc.cscs.ch/v1"
EMBEDDING_MODEL = "Snowflake/snowflake-arctic-embed-l-v2.0"
MODEL_NAME = "moonshotai/Kimi-K2.5-SDSC"  # Strongest model listed in the API example.
# Fast alternative: "zai-org/GLM-4.7-Flash"
# Open model alternatives: "swiss-ai/Apertus-8B-Instruct-2509", "swiss-ai/Apertus-70B-Instruct-2509"

client = OpenAI(
    base_url=API_BASE_URL,
    api_key=os.getenv("CSCS_API_KEY"),
)

if not QA_CSV.exists():
    QA_CSV.write_text(EMBEDDED_QA_CSV, encoding="utf-8")
    print("qa_set.csv was missing, so the embedded QA set was written to:", QA_CSV)

print("Working directory:", WORK_DIR)
print("qa_set.csv exists:", QA_CSV.exists())
print("API key configured:", bool(os.getenv("CSCS_API_KEY")))

Working directory: /home/renku/work/Durham-Hackathon-2026-w2t1
qa_set.csv exists: True
API key configured: True


## 2. Load Markdown Corpus

In [36]:
IMAGE_PATTERN = re.compile(r"!\[[^\]]*\]\([^)]+\)")
WORD_PATTERN = re.compile(r"\S+")


def clean_markdown(text: str) -> str:
    text = IMAGE_PATTERN.sub(" ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"[ \t]+", " ", text)
    return text


markdown_files = sorted(
    path for path in WORK_DIR.glob("*.md")
    if path.name.lower() not in {"readme.md"}
)

if not markdown_files:
    raise FileNotFoundError(
        "No .md files found. Put the PDF-derived Markdown files in the same folder as this notebook."
    )

print(f"Found {len(markdown_files)} Markdown files:")
for path in markdown_files:
    print("-", path.name)

Found 4 Markdown files:
- Web Version _E-Government Survey 2024 11102024.md
- World_Inequality_Report_2026.md
- natural-catastrophe-and-climate-report-2023.md
- swissre_sigma-1_2024_english.md


## 3. Chunk the Corpus

In [37]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: int
    source: str
    chunk_index: int
    start_word: int
    end_word: int
    text: str


def chunk_text(source: str, text: str, chunk_words: int, overlap_words: int, start_id: int) -> list[Chunk]:
    if overlap_words >= chunk_words:
        raise ValueError("Overlap must be smaller than chunk size.")

    words = WORD_PATTERN.findall(clean_markdown(text))
    if not words:
        return []

    chunks = []
    step = chunk_words - overlap_words
    for chunk_index, start in enumerate(range(0, len(words), step)):
        end = min(start + chunk_words, len(words))
        chunks.append(
            Chunk(
                chunk_id=start_id + len(chunks),
                source=source,
                chunk_index=chunk_index,
                start_word=start,
                end_word=end,
                text=" ".join(words[start:end]),
            )
        )
        if end == len(words):
            break
    return chunks


chunks: list[Chunk] = []
next_id = 1
for md_path in markdown_files:
    text = md_path.read_text(encoding="utf-8", errors="replace")
    file_chunks = chunk_text(md_path.name, text, CHUNK_WORDS, OVERLAP_WORDS, next_id)
    chunks.extend(file_chunks)
    next_id += len(file_chunks)

print(f"Created {len(chunks)} chunks.")
pd.DataFrame([c.__dict__ for c in chunks[:5]])

Created 1044 chunks.


,chunk_id,source,chunk_index,start_word,end_word,text
0,1,Web Version _E-Government Survey 2024 11102024.md,0,0,250,## **E-Government Survey 2024** Accelerating D...
1,2,Web Version _E-Government Survey 2024 11102024.md,1,200,450,‘country’ and ‘economy’ as used in this Report...
2,3,Web Version _E-Government Survey 2024 11102024.md,2,400,650,"or its senior management, or of the experts wh..."
3,4,Web Version _E-Government Survey 2024 11102024.md,3,600,850,2024 UN E-GovErNmENt SUrvEy iv PREFAcE ## **Pr...
4,5,Web Version _E-Government Survey 2024 11102024.md,4,800,1050,is crucial for comprehensive digital transform...


## 4. Build API Embeddings and Save the RAG Database

In [38]:
def embed_text(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    response = client.embeddings.create(
        model=model,
        input=text,
    )
    return response.data[0].embedding


def embed_texts(texts: list[str], model: str = EMBEDDING_MODEL, batch_size: int = 16) -> np.ndarray:
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding chunks"):
        batch = texts[start : start + batch_size]
        try:
            response = client.embeddings.create(
                model=model,
                input=batch,
            )
            vectors.extend(item.embedding for item in response.data)
        except Exception:
            # Some OpenAI-compatible embedding endpoints only accept one input at a time.
            for text in batch:
                vectors.append(embed_text(text, model=model))

    matrix = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


if not os.getenv("CSCS_API_KEY"):
    raise RuntimeError("Set CSCS_API_KEY before creating API embeddings.")

chunk_texts = [chunk.text for chunk in chunks]
chunk_matrix = embed_texts(chunk_texts, batch_size=16)

print("Embedding model:", EMBEDDING_MODEL)
print("Embedding matrix shape:", chunk_matrix.shape)

Embedding chunks: 100%|██████████| 66/66 [00:08<00:00,  7.60it/s]


Embedding model: Snowflake/snowflake-arctic-embed-l-v2.0
Embedding matrix shape: (1044, 1024)


In [39]:
def save_database(db_path: Path) -> None:
    if db_path.exists():
        db_path.unlink()

    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(
            """
            CREATE TABLE metadata (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL
            );

            CREATE TABLE chunks (
                id INTEGER PRIMARY KEY,
                source TEXT NOT NULL,
                chunk_index INTEGER NOT NULL,
                start_word INTEGER NOT NULL,
                end_word INTEGER NOT NULL,
                text TEXT NOT NULL,
                embedding BLOB NOT NULL
            );

            CREATE INDEX idx_chunks_source ON chunks(source);
            """
        )
        metadata = {
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "chunk_words": CHUNK_WORDS,
            "overlap_words": OVERLAP_WORDS,
            "similarity_threshold": SIMILARITY_THRESHOLD,
            "max_context_chunks": MAX_CONTEXT_CHUNKS,
            "min_context_chunks": MIN_CONTEXT_CHUNKS,
            "embedding_backend": EMBEDDING_MODEL,
            "embedding_dimension": int(chunk_matrix.shape[1]),
            "chunk_count": len(chunks),
            "sources": sorted({chunk.source for chunk in chunks}),
        }
        conn.execute(
            "INSERT INTO metadata(key, value) VALUES (?, ?)",
            ("index", json.dumps(metadata, indent=2)),
        )
        conn.executemany(
            """
            INSERT INTO chunks(id, source, chunk_index, start_word, end_word, text, embedding)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            [
                (
                    chunk.chunk_id,
                    chunk.source,
                    chunk.chunk_index,
                    chunk.start_word,
                    chunk.end_word,
                    chunk.text,
                    chunk_matrix[row].astype(np.float32).tobytes(),
                )
                for row, chunk in enumerate(chunks)
            ],
        )
        conn.commit()
    finally:
        conn.close()


save_database(DB_PATH)
print("Saved database:", DB_PATH)

Saved database: /home/renku/work/Durham-Hackathon-2026-w2t1/rag_api_embedding_database.sqlite


## 5. Retrieval With Cosine Similarity Threshold

In [40]:
def retrieve_chunks(
    question: str,
    threshold: float = SIMILARITY_THRESHOLD,
    max_chunks: int = MAX_CONTEXT_CHUNKS,
    min_chunks: int = MIN_CONTEXT_CHUNKS,
) -> list[dict]:
    query_vector = np.asarray(embed_text(question), dtype=np.float32)
    query_vector = query_vector / max(float(np.linalg.norm(query_vector)), 1e-12)
    scores = chunk_matrix @ query_vector
    ranked_indices = np.argsort(scores)[::-1]

    selected = [
        {
            "chunk_id": chunks[i].chunk_id,
            "source": chunks[i].source,
            "chunk_index": chunks[i].chunk_index,
            "score": float(scores[i]),
            "text": chunks[i].text,
        }
        for i in ranked_indices
        if scores[i] >= threshold
    ][:max_chunks]

    # Fallback: if the threshold is too strict for a short/narrow question,
    # still provide the best few chunks so the model can answer or say not found.
    if len(selected) < min_chunks:
        fallback = [
            {
                "chunk_id": chunks[i].chunk_id,
                "source": chunks[i].source,
                "chunk_index": chunks[i].chunk_index,
                "score": float(scores[i]),
                "text": chunks[i].text,
            }
            for i in ranked_indices[:min_chunks]
        ]
        seen = {item["chunk_id"] for item in selected}
        selected.extend(item for item in fallback if item["chunk_id"] not in seen)

    return selected[:max_chunks]


sample_question = "How much higher are the 2023 insured losses than the previous 10 year average?"
sample_chunks = retrieve_chunks(sample_question)
[(c["score"], c["source"], c["chunk_index"]) for c in sample_chunks]

[(0.6496871709823608, 'swissre_sigma-1_2024_english.md', 12),
 (0.632580041885376, 'swissre_sigma-1_2024_english.md', 19),
 (0.6285090446472168, 'swissre_sigma-1_2024_english.md', 25),
 (0.6272820234298706, 'natural-catastrophe-and-climate-report-2023.md', 25),
 (0.6262193918228149, 'natural-catastrophe-and-climate-report-2023.md', 21),
 (0.6144300103187561, 'swissre_sigma-1_2024_english.md', 20)]

## 6. API Answer Generation

In [53]:
SYSTEM_PROMPT = """You are a careful RAG question-answering assistant.
Use only the supplied context.
Answer concisely, preferably in one or two sentences.
If the context does not contain the answer, write exactly: Not found in the retrieved context.
When relevant, include exact years, percentages, counts, names, or figure numbers.
Do not invent citations. Do not mention chunks unless asked.
"""


def format_context(retrieved: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved, start=1):
        text = chunk["text"]
        parts.append(
            f"[Context {i}] source={chunk['source']} chunk={chunk['chunk_index']} cosine={chunk['score']:.3f}\n{text}"
        )
    return "\n\n".join(parts)


def extract_chat_message_text(message) -> str:
    """Handle OpenAI-compatible servers that do not always use message.content."""
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        return content.strip()
    if isinstance(content, list):
        pieces = []
        for item in content:
            if isinstance(item, dict):
                pieces.append(str(item.get("text") or item.get("content") or ""))
            else:
                pieces.append(str(getattr(item, "text", "") or getattr(item, "content", "")))
        joined = "\n".join(piece for piece in pieces if piece.strip()).strip()
        if joined:
            return joined

    data = message.model_dump() if hasattr(message, "model_dump") else dict(message)
    for key in ("reasoning_content", "reasoning", "output_text", "refusal"):
        value = data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
        if isinstance(value, dict):
            text = value.get("text") or value.get("content")
            if isinstance(text, str) and text.strip():
                return text.strip()

    return "Not found in the retrieved context."


def answer_question(
    question: str,
    retrieved: list[dict] | None = None,
    model: str = MODEL_NAME,
) -> tuple[str, list[dict]]:
    if retrieved is None:
        retrieved = retrieve_chunks(question)
    context = format_context(retrieved)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:",
            },
        ],
        temperature=0.0,
    )
    return extract_chat_message_text(response.choices[0].message), retrieved

In [42]:
# Single-question test.
# This cell requires CSCS_API_KEY to be configured.

if not os.getenv("CSCS_API_KEY"):
    print("Set CSCS_API_KEY before running API calls.")
else:
    evidence = retrieve_chunks(sample_question)
    answer, evidence = answer_question(sample_question, evidence)
    print("Question:", sample_question)
    print("Answer:", answer)
    print("Evidence:")
    for item in evidence:
        print(f"- {item['source']}#{item['chunk_index']} cosine={item['score']:.3f}")

Question: How much higher are the 2023 insured losses than the previous 10 year average?
Answer: Not found in the retrieved context.
Evidence:
- swissre_sigma-1_2024_english.md#12 cosine=0.650
- swissre_sigma-1_2024_english.md#19 cosine=0.633
- swissre_sigma-1_2024_english.md#25 cosine=0.629
- natural-catastrophe-and-climate-report-2023.md#25 cosine=0.627
- natural-catastrophe-and-climate-report-2023.md#21 cosine=0.626
- swissre_sigma-1_2024_english.md#20 cosine=0.614


## 7. Load QA CSV

In [43]:
def read_qa_set(path: Path) -> pd.DataFrame:
    # qa_set.csv uses semicolons: question;answer
    df = pd.read_csv(path, sep=";", keep_default_na=False)
    if "question" not in df.columns:
        raise ValueError("qa_set.csv must contain a 'question' column.")
    if "answer" not in df.columns:
        df["answer"] = ""
    df["question"] = df["question"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    return df[df["question"] != ""].reset_index(drop=True)


qa_df = read_qa_set(QA_CSV)
print("Questions:", len(qa_df))
print("Existing answers:", int((qa_df["answer"] != "").sum()))
qa_df.head()

Questions: 40
Existing answers: 5


,question,answer
0,What year where there most casualties from man...,"2002. In that year more than 10,000 casualties..."
1,In what year did the quantity of man-made disa...,Man-made disasters peaked in 2005
2,Between 1970 and 2023 what year in the data sh...,2023. There were 218 instances of natural cat...
3,What year between 1994 and 2023 had the most h...,2011 with 6.
4,How much higher are the 2023 insured losses th...,They are higher by 21%.


## 8. Answer All Questions

In [56]:
def answer_qa_dataframe(
    df: pd.DataFrame,
    preserve_existing: bool = True,
    sleep_seconds: float = 0.2,
) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        question = row["question"]
        existing_answer = row["answer"]

        retrieved = retrieve_chunks(question)
        if preserve_existing and existing_answer:
            answer = existing_answer
        else:
            answer, retrieved = answer_question(question, retrieved)
            time.sleep(sleep_seconds)

        rows.append(
            {
                "question": question,
                "answer": answer,
                "retrieved_sources": " | ".join(
                    f"{item['source']}#{item['chunk_index']} ({item['score']:.3f})"
                    for item in retrieved
                ),
                "retrieved_chunk_ids": ",".join(str(item["chunk_id"]) for item in retrieved),
            }
        )
    return pd.DataFrame(rows)


if not os.getenv("CSCS_API_KEY"):
    print("Set CSCS_API_KEY before answering the full QA set.")
else:
    results_df = answer_qa_dataframe(qa_df, preserve_existing=True)
    results_df.to_csv(OUTPUT_CSV, sep=";", index=False)
    print("Wrote:", OUTPUT_CSV)
    display(results_df)

100%|██████████| 40/40 [18:04<00:00, 27.11s/it]

Wrote: /home/renku/work/Durham-Hackathon-2026-w2t1/qa_set_api_rag_answers.csv


,question,answer,retrieved_sources,retrieved_chunk_ids
0,What year where there most casualties from man...,"2002. In that year more than 10,000 casualties...",swissre_sigma-1_2024_english.md#72 (0.579) | s...,"1032,1033,1034,953,1039,952"
1,In what year did the quantity of man-made disa...,Man-made disasters peaked in 2005,swissre_sigma-1_2024_english.md#72 (0.609) | s...,"1032,1033,979,1034,823,953"
2,Between 1970 and 2023 what year in the data sh...,2023. There were 218 instances of natural cat...,swissre_sigma-1_2024_english.md#72 (0.642) | s...,"1032,979,953,823,1033,836"
3,What year between 1994 and 2023 had the most h...,2011 with 6.,natural-catastrophe-and-climate-report-2023.md...,"836,982,823,965,835,981"
4,How much higher are the 2023 insured losses th...,They are higher by 21%.,swissre_sigma-1_2024_english.md#12 (0.650) | s...,"972,979,985,843,839,980"
5,Which figure shows the trend in insured losses...,Figure 4 shows the trend in insured losses fro...,swissre_sigma-1_2024_english.md#20 (0.637) | n...,"980,843,985,979,839,972"
6,"Before 2023, what was the highest year on reco...",Not found in the retrieved context.,natural-catastrophe-and-climate-report-2023.md...,"899,990,976,962,978,967"
7,What regions does the Swiss Re report on natur...,The Swiss Re report splits the US into six reg...,swissre_sigma-1_2024_english.md#34 (0.661) | s...,"994,1037,990,984,992,993"
8,What is the highest Benefit to Cost ratio buil...,Earthquake-resistant building codes have the h...,swissre_sigma-1_2024_english.md#65 (0.571) | s...,"1025,1021,1024,1039,984,1043"
9,According to the Swiss Re Institute report on ...,Not found in the retrieved context.,swissre_sigma-1_2024_english.md#20 (0.683) | s...,"980,960,979,984,1040,840"


In [57]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

display(results_df)

,question,answer,retrieved_sources,retrieved_chunk_ids
0,What year where there most casualties from man-made disasters in the recorded data?,"2002. In that year more than 10,000 casualties are ascribed to man-made catastrophes.",swissre_sigma-1_2024_english.md#72 (0.579) | swissre_sigma-1_2024_english.md#73 (0.571) | swissre_sigma-1_2024_english.md#74 (0.565) | natural-catastrophe-and-climate-report-2023.md#135 (0.527) | swissre_sigma-1_2024_english.md#79 (0.484) | natural-catastrophe-and-climate-report-2023.md#134 (0.475),"1032,1033,1034,953,1039,952"
1,In what year did the quantity of man-made disasters peak in the recorded data between 1970 and 2023?,Man-made disasters peaked in 2005,swissre_sigma-1_2024_english.md#72 (0.609) | swissre_sigma-1_2024_english.md#73 (0.553) | swissre_sigma-1_2024_english.md#19 (0.552) | swissre_sigma-1_2024_english.md#74 (0.551) | natural-catastrophe-and-climate-report-2023.md#5 (0.542) | natural-catastrophe-and-climate-report-2023.md#135 (0.540),"1032,1033,979,1034,823,953"
2,Between 1970 and 2023 what year in the data shows the largest number of natural catastrophes?,2023. There were 218 instances of natural catastrophes.,swissre_sigma-1_2024_english.md#72 (0.642) | swissre_sigma-1_2024_english.md#19 (0.608) | natural-catastrophe-and-climate-report-2023.md#135 (0.608) | natural-catastrophe-and-climate-report-2023.md#5 (0.604) | swissre_sigma-1_2024_english.md#73 (0.595) | natural-catastrophe-and-climate-report-2023.md#18 (0.593),"1032,979,953,823,1033,836"
3,What year between 1994 and 2023 had the most high severity ($5 billion in damages or more) natural catastrophes?,2011 with 6.,natural-catastrophe-and-climate-report-2023.md#18 (0.655) | swissre_sigma-1_2024_english.md#22 (0.647) | natural-catastrophe-and-climate-report-2023.md#5 (0.642) | swissre_sigma-1_2024_english.md#5 (0.640) | natural-catastrophe-and-climate-report-2023.md#17 (0.632) | swissre_sigma-1_2024_english.md#21 (0.627),"836,982,823,965,835,981"
4,How much higher are the 2023 insured losses than the previous 10 year average?,They are higher by 21%.,swissre_sigma-1_2024_english.md#12 (0.650) | swissre_sigma-1_2024_english.md#19 (0.633) | swissre_sigma-1_2024_english.md#25 (0.629) | natural-catastrophe-and-climate-report-2023.md#25 (0.627) | natural-catastrophe-and-climate-report-2023.md#21 (0.626) | swissre_sigma-1_2024_english.md#20 (0.614),"972,979,985,843,839,980"
5,"Which figure shows the trend in insured losses over data from 1994 to 2023? In this figure, what is the highest insured loss year on record?",Figure 4 shows the trend in insured losses from 1994 to 2023. The highest insured loss year on record in this figure is **2022** with **USD 133 billion** (inflation-adjusted).,swissre_sigma-1_2024_english.md#20 (0.637) | natural-catastrophe-and-climate-report-2023.md#25 (0.630) | swissre_sigma-1_2024_english.md#25 (0.623) | swissre_sigma-1_2024_english.md#19 (0.619) | natural-catastrophe-and-climate-report-2023.md#21 (0.613) | swissre_sigma-1_2024_english.md#12 (0.605),"980,843,985,979,839,972"
6,"Before 2023, what was the highest year on record for European Severe Convective Storm losses?",Not found in the retrieved context.,natural-catastrophe-and-climate-report-2023.md#81 (0.625) | swissre_sigma-1_2024_english.md#30 (0.620) | swissre_sigma-1_2024_english.md#16 (0.601) | swissre_sigma-1_2024_english.md#2 (0.598) | swissre_sigma-1_2024_english.md#18 (0.594) | swissre_sigma-1_2024_english.md#7 (0.588),"899,990,976,962,978,967"
7,What regions does the Swiss Re report on natural catastrophes split the US into for severe convective storm risk?,"The Swiss Re report splits the US into six regions for severe convective storm risk: **Rockies, Midwest, West, Northeast, Central, and Southeast** (see Figure 12).",swissre_sigma-1_2024_english.md#34 (0.661) | swissre_sigma-1_2024_english.md#77 (0.610) | swissre_sigma-1_2024_english.md#30 (0.584) | swissre_sigma-1_2024_english.md#24 (0.568) | swissre_sigma-1_2024_english.md#32 (0.553)

In [58]:
TEAM_NAME = "Matterhorn"

In [59]:
import requests

BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch"
leaderboard_endpoint = f"{BASE_URL}/api/v1/leaderboard"
submit_endpoint = f"{BASE_URL}/api/v1/submit"


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(leaderboard_endpoint).json()
    return pd.DataFrame(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str) -> pd.Series:
    if user == "your_team_name" or user == "":
        raise ValueError("Please set your team name in the 'TEAM_NAME' variable.")
    predictions = df[["question", "answer"]].to_dict(orient="records")
    submission = {"predictions": predictions}
    response = requests.post(submit_endpoint, json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        response.raise_for_status()
    print(response.json())
    return pd.Series(response.json())

In [60]:
submit(results_df, user=TEAM_NAME, token="cant_be_empty")

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [62]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics
0,Durham Teddy Bear,1,2,2026-06-09T15:04:42.218812,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.5, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0], 'difficulty': [2.0, 2.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 1.5, 1.0], 'total_score': 0.5963302752293578, 'questions_answered': 34, 'total_questions': 40}"
1,Alphubel,2,6,2026-06-09T15:34:17.757739,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.5, 1.0, 0.0, 0.5, 1.0, 1.0, 0.0, 1.0, 0.5, 1.0, 0.0, 0.0, 0.5, 0.0, 0.5, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5], 'difficulty': [2.0, 2.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 1.5, 1.0], 'total_score': 0.46788990825688076, 'questions_answered': 30, 'total_questions': 40}"
2,Matterhorn,3,6,2026-06-09T16:13:30.257828,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.5, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.5, 0.0, 1.0], 'difficulty': [2.0, 2.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 1.5, 1.0], 'total_score': 0.4036697247706422, 'questions_answered': 23, 'total_questions': 40}"
3,Zinalrothorn,4,63,2026-06-09T12:59:55.336793,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.5, 1.0, 1.0, 0.5, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 0.0, 0.5, 1.0, 1.0, 1.0, 1.0, 0.5, 0.5, 0.0, 0.0], 'difficulty': [2.0, 2.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 1.5, 1.0], 'total_score': 0.3669724770642202, 'questions_answered': 23, 'total_questions': 40}"
4,Allalinhorn,5,47,2026-06-09T16:02:28.831988,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 1.0, 1.0, 0.5, 1.0, 0.0, 0.0, 0.0, 0.5], 'difficulty': [2.0, 2.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 1.5, 1.0], 'total_score': 0.3119266055045872, 'questions_answered': 19, 'total_questions': 40}"


## 9. Optional: Regenerate Existing Answers Too

In [45]:
# Uncomment to regenerate every row, including rows that already had answers.
#
# results_all_regenerated = answer_qa_dataframe(qa_df, preserve_existing=False)
# results_all_regenerated.to_csv(WORK_DIR / "qa_set_api_rag_answers_regenerated.csv", sep=";", index=False)
# display(results_all_regenerated.head())

## 10. Inspect Low-Retrieval Questions

In [46]:
# This helps tune SIMILARITY_THRESHOLD and MAX_CONTEXT_CHUNKS.
# With Snowflake/snowflake-arctic-embed-l-v2.0, start around 0.30.
# If many top scores are below 0.30, lower SIMILARITY_THRESHOLD to 0.25.
# If answers need more context, raise MAX_CONTEXT_CHUNKS to 8.

diagnostics = []
for question in qa_df["question"]:
    retrieved = retrieve_chunks(question)
    diagnostics.append(
        {
            "question": question,
            "top_score": retrieved[0]["score"] if retrieved else 0.0,
            "num_chunks_selected": len(retrieved),
            "top_source": retrieved[0]["source"] if retrieved else "",
            "top_chunk": retrieved[0]["chunk_index"] if retrieved else "",
        }
    )

diag_df = pd.DataFrame(diagnostics).sort_values("top_score")
display(diag_df.head(10))

,question,top_score,num_chunks_selected,top_source,top_chunk
18,Of the questions highlighted from the Member S...,0.444222,6,World_Inequality_Report_2026.md,12
16,Between what years was the EGDI revision 3.0 a...,0.527681,6,Web Version _E-Government Survey 2024 11102024.md,43
24,In what percentage of countries in Oceania is ...,0.537075,6,Web Version _E-Government Survey 2024 11102024.md,185
26,In what percentage of countries in the America...,0.541763,6,Web Version _E-Government Survey 2024 11102024.md,178
25,What region of the world contains the countrie...,0.546052,6,Web Version _E-Government Survey 2024 11102024.md,214
8,What is the highest Benefit to Cost ratio buil...,0.570719,6,swissre_sigma-1_2024_english.md,65
35,What two departments of the UK govenrment have...,0.572840,6,Web Version _E-Government Survey 2024 11102024.md,323
12,What is the lower bound of dead or missing whi...,0.576214,6,swissre_sigma-1_2024_english.md,82
0,What year where there most casualties from man...,0.579182,6,swissre_sigma-1_2024_english.md,72
34,How has Japan named their initiative for remov...,0.587917,6,Web Version _E-Government Survey 2024 11102024.md,294


In [66]:
q = qa_df.loc[23, "question"]

retrieved = retrieve_chunks(q)

print("Question:", q)
print("\nRetrieved chunks:")
for item in retrieved:
    print(f"- {item['source']}#{item['chunk_index']} cosine={item['score']:.3f}")
    print(item["text"][:500])
    print()

print("\nFull context sent to model:")
print(format_context(retrieved)[:50000])

answer, evidence = answer_question(q, retrieved)
print("\nAnswer:")
print(answer)

Question: What percentage of european countries support filing income tax online?

Retrieved chunks:
- Web Version _E-Government Survey 2024 11102024.md#176 cosine=0.627
birth certificate, and filing company taxes. The electronic submission of business taxes is offered by more countries than the online submission of income taxes, which is a departure from 2022. Tax-filing services are offered more frequently to businesses (157 countries) than to individuals (152 countries for inco ~~me tax and 147 countr~~ ies ~~for Va~~ lue ~~Added~~ Tax, or VAT). Among the least ~~offered onl~~ i ~~ne services~~ a ~~re changing an address (84 countri~~ e ~~s) and regis~~ terin

- Web Version _E-Government Survey 2024 11102024.md#230 cosine=0.542
one of the online services assessed for the 2024 Survey remains at 189 (98 per cent). The global average number of online services offered relative to the number of services assessed has risen from 16 out of 22 in 2022 to 18 out of 25 in 2024. The online prov